# 34 · 生成评估：Faithfulness / 框架 / 基准

> 检索再好，回答满嘴跑火车照样完蛋。生成质量用**忠实度与相关性**衡量——最主流的方式是**让 LLM 当裁判**（LLM-as-a-Judge）。

**本文件覆盖知识点**：Faithfulness / Answer Relevance / Context Relevance / LLM-as-a-Judge / RAGAS / DeepEval / TruLens / LangSmith / 数据集 MS MARCO / BEIR

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 三个最核心的生成指标

| 指标 | 英文 | 问的是 |
|------|------|--------|
| **忠实度** | Faithfulness | 回答是否**忠于检索到的上下文**、没瞎编 |
| **答案相关性** | Answer Relevance | 回答是否**答在点上**（不是废话也相关） |
| **上下文相关性** | Context Relevance | 检索到的上下文**是否用上了/有没有噪声** |

> Faithfulness 是防幻觉的关键闸门：回答里的每个论断都应能溯源到某个检索片段。

## 先用大白话讲一遍（三个指标到底在算什么）

先看一次具体的问答，后面所有算法都是围着它转：

- **问题**：星云支持私有化部署吗？
- **检索到的上下文**：`星云支持公有云与私有化两种部署方式。`
- **模型回答**：`星云既支持公有云也支持私有化部署，而且性能是友商两倍。`

**1）Faithfulness（忠实度）= 回答里的话，有多少能在检索到的上下文里找到依据**

做法：把回答切成一条条「可单独核对的小断言」，逐条问一句"上下文里说了吗？"

| 断言 | 上下文里说了吗 |
|------|----------------|
| 星云支持公有云部署 | 说了 ✅ |
| 星云支持私有化部署 | 说了 ✅ |
| 性能是友商两倍 | 从没说过 ❌ ← **这就是幻觉** |

算：**有依据的 2 条 ÷ 一共 3 条 = 0.67**。注意分母是**断言条数**，不是句子数 —— 切得越细，越能指出到底是哪一句话在瞎编。

**2）Answer Relevance（答案相关性）= 光看这个回答，能不能反推出原问题**

做法：不比对上下文，而是把**回答**交给模型问一句"这段话像是在回答什么问题？"，让它生成几个问题，再和原问题比语义相似度（余弦相似度，越接近 1 越像），取平均。

- 答得越切题 → 反推出的问题越接近原问题 → 分越高；
- 答得越空泛、越跑题 → 反推的问题越东一榔头西一棒 → 分越低。

要注意：它只看「回答 ↔ 问题」这一对，**完全不看上下文**。所以"编得很切题"也能拿高分 —— 这正是它必须和 Faithfulness 配对使用的原因。

**3）Context Relevance（上下文相关性）= 检索来的资料里，有多少是真的有用的**

做法：把上下文切成句子，逐句判断"跟这个问题有关吗"，算相关句子占的比例。

再进一步，**ContextPrecision@K 还看顺序**：同样是 1 条有用，排在第 1 位比排在第 3 位得分更高 —— 因为它算的是"走到每一句时的命中率"再按是否相关加权平均。

**4）三个分数怎么合成一个总分**

最常用的是「RAG 三件套」（RAG Triad）：三个维度各自的平均分再取平均；也可以取**最小值**当红线 —— 任何一维不及格就判整体不通过，适合挂到线上做告警。

> 下面代码块会把这三个指标的中间数字都打印出来（联网时答案相关性用真实 embedding 算），对照着看一遍，再看下一节的公式就顺了。

## 精确计算公式

> **约定**：$Q$ 为评测问题集；$q$ 一个问题；$A$ 为模型回答；$C=\{c_1,\dots,c_m\}$ 为检索到的上下文片段；裁判模型记作 $\mathcal{J}$；所有指标归一到 $[0,1]$，**整集得分 = 各问题得分的算术平均** $S=\frac{1}{|Q|}\sum_{q\in Q}s(q)$。

**1) Faithfulness（忠实度）：回答有没有瞎编**
把回答拆成**原子断言** $\{a_1,\dots,a_n\}$（每条只含一个事实），逐条判断能否由上下文 $C$ 推出：
$$\mathrm{supp}(a)=\mathbb{1}\bigl[\,a\ \text{可由}\ C\ \text{推出}\,\bigr]$$
$$\mathrm{Faithfulness}=\frac{1}{n}\sum_{i=1}^{n}\mathrm{supp}(a_i)=\frac{\bigl|\{a_i:\mathrm{supp}(a_i)=1\}\bigr|}{n}\in[0,1]$$
分母是**断言条数**而非句子数；拆得越细，越能定位幻觉出在哪条论断（本课代码即此式：3 条断言支持 2 条 → 0.67）。

**2) Answer Relevance（答案相关性）：回答有没有答在点上**
RAGAS 的“反向生成”定义——由回答 $A$ 反向生成 $n$ 个候选问题 $\{q_1,\dots,q_n\}$（“这个回答像是在回答什么问题”），与原问题做语义相似度取均值：
$$\mathrm{AnswerRelevance}(q)=\frac{1}{n}\sum_{i=1}^{n}\cos\bigl(\mathbf{e}(q),\ \mathbf{e}(q_i)\bigr)$$
其中 $\mathbf{e}(\cdot)$ 为 embedding 向量，$\cos$ 为余弦相似度。回答越空泛，反推出的问题越分散，均值越低。

**3) Context Relevance（上下文相关性）：检索来的上下文有没有用**
把上下文拆成句子 $\{s_1,\dots,s_t\}$，逐句判断与问题是否相关：
$$\mathrm{ContextRelevance}=\frac{\bigl|\{s_j:\ \mathrm{relevant}(s_j,q)=1\}\bigr|}{t}\in[0,1]$$
**排序敏感版**（RAGAS 的 context precision@K，奖励相关片段排在前面）：
$$\mathrm{ContextPrecision@K}=\frac{\sum_{k=1}^{K}\bigl(\mathrm{Precision@}k\times v_k\bigr)}{\sum_{k=1}^{K}v_k},\qquad v_k=\mathbb{1}\bigl[\text{第 }k\text{ 个片段相关}\bigr]$$
$$\text{其中}\quad \mathrm{Precision@}k=\frac{1}{k}\sum_{j=1}^{k}v_j$$

**4) LLM-as-a-Judge 的打分与聚合**
裁判按评分卡对三维分别给分 $s\in[0,1]$（若给 1~5 分则线性归一 $\frac{s-1}{4}$）：
$$s_{\text{faithful}},\,s_{\text{answer\_rel}},\,s_{\text{context\_rel}}=\mathcal{J}(q,\,C,\,A)$$
整集得分 $\bar{s}=\frac{1}{|Q|}\sum_{q}s(q)$。RAG 三件套（RAG Triad）的综合分可取
$$S_{\text{triad}}=\frac{1}{3}\Bigl(\overline{s_{\text{faithful}}}+\overline{s_{\text{answer\_rel}}}+\overline{s_{\text{context\_rel}}}\Bigr)$$
或按业务取三者**最小值**当红线（任一维不达标即判不通过）：
$$S_{\min}=\min\bigl(\overline{s_{\text{faithful}}},\ \overline{s_{\text{answer\_rel}}},\ \overline{s_{\text{context\_rel}}}\bigr)$$

> 评测要固定四件事才可比：**裁判模型、评分卡 prompt、温度（建议 0~0.1）、输出解析方式**。裁判本身有位置偏好与长度偏好，需固定资料顺序、统一答案格式。

In [2]:
# 手算演示：三个指标逐个把中间量打印出来（Faithfulness/Context 相关为纯算术，答案相关性联网时真调 embedding）
import numpy as np
from dotenv import load_dotenv
load_dotenv()
import os

_EMB_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_EMB = bool(_EMB_KEY) and '你的' not in _EMB_KEY

q = '星云支持私有化部署吗'
ctx = ['星云支持公有云部署，星云支持私有化部署。']
ans = '星云既支持公有云也支持私有化部署，而且性能是友商两倍。'
claims = ['星云支持公有云部署', '星云支持私有化部署', '性能是友商两倍']

# ---------- 1) Faithfulness ----------
print('=== 1) Faithfulness：回答里的断言，有几条能在上下文里找到依据 ===')
for c in claims:
    ok = any(c in s for s in ctx)
    print('  断言「%s」 → %s' % (c, '上下文里有依据 ✅' if ok else '上下文里没有 → 判为幻觉 ❌'))
sup = sum(1 for c in claims if any(c in s for s in ctx))
print('  算式：有依据 %d ÷ 断言总数 %d = %.2f' % (sup, len(claims), sup / len(claims)))
print('  （这里用"字符串包含"做教学版判定；生产上由裁判模型逐条语义判断）')

# ---------- 2) Answer Relevance ----------
print('')
print('=== 2) Answer Relevance：把回答反推成问题，再和原问题比相似度 ===')
gen_qs = ['星云支持哪些部署方式？', '星云的部署方式是怎样的？', '星云的产品价格是多少？']

def _toy_cos(a, b):
    """无 Key 时的玩具相似度：按字符集合重叠估算，只为演示算式，真实实现必须用 embedding。"""
    A, B = set(a), set(b)
    return len(A & B) / (np.sqrt(len(A)) * np.sqrt(len(B))) if A and B else 0.0

sims = None
if _HAS_EMB:
    try:
        from dashscope import TextEmbedding
        r = TextEmbedding.call(model='text-embedding-v3', input=[q] + gen_qs, api_key=_EMB_KEY)
        vecs = [e['embedding'] for e in sorted(r.output['embeddings'], key=lambda e: e['text_index'])]
        v = np.array(vecs, dtype=float)
        v = v / np.linalg.norm(v, axis=1, keepdims=True)
        sims = [float(v[0] @ v[i]) for i in range(1, len(v))]
        print('  （真实调用 text-embedding-v3 计算的余弦相似度）')
    except Exception as e:
        print('  embedding 调用失败（%s），改用玩具相似度演示算式' % e)
if sims is None:
    sims = [_toy_cos(q, g) for g in gen_qs]
    print('  （未配置 Key：下面是字符重叠的玩具相似度，仅用于看懂算式）')
for g, s in zip(gen_qs, sims):
    print('  反推问题「%s」 与原问题的相似度 = %.3f' % (g, s))
print('  算式：相似度取平均 = %.3f   ← 最后那条问的是"价格"，跟部署无关，通常最低、把均分拉下来'
      % float(np.mean(sims)))

# ---------- 3) Context Relevance ----------
print('')
print('=== 3) Context Relevance：上下文里的句子，有几条真的跟问题相关 ===')
sents = [('星云支持公有云与私有化两种部署方式', 1), ('星云成立于 2015 年', 0)]
for s, v_ in sents:
    print('  句子「%s」 → %s' % (s, '相关 ✅' if v_ else '跑题 ❌'))
rel_cnt = sum(v_ for _, v_ in sents)
print('  算式：相关句数 %d ÷ 句子总数 %d = %.2f' % (rel_cnt, len(sents), rel_cnt / len(sents)))
print('  （上面这句是另一段上下文，用来演示"整段里混进了无关句子"；相关/跑题按人工判定写死，生产上由裁判模型逐句判）')

# ---------- 3b) ContextPrecision@K ----------
print('')
print('=== 3b) ContextPrecision@K：还看顺序 —— 有用的片段排得越前越高 ===')
v = [1, 0, 1]        # 三个检索片段的判定：第 1 个相关、第 2 个跑题、第 3 个相关
num = 0.0
for k_, vk in enumerate(v, 1):
    p_at_k = sum(v[:k_]) / k_          # 走到第 k 个时，命中率是多少
    num += p_at_k * vk
    print('  第 %d 个片段 相关=%d → Precision@%d = %d/%d = %.2f' % (k_, vk, k_, sum(v[:k_]), k_, p_at_k))
print('  算式：ContextPrecision@3 = %.2f ÷ 相关片段数 %d = %.2f' % (num, sum(v), num / sum(v)))
v_best = [1, 1, 0]  # 同样 2 条相关片段，只是排在了第 1、2 位
num_best = sum(sum(v_best[:i]) / i * v_best[i - 1] for i in range(1, len(v_best) + 1))
print('  对照：同样 2 条相关片段若排在第 1、2 位 → ContextPrecision@3 = %.2f（排得越靠前越值钱）'
      % (num_best / sum(v_best)))

=== 1) Faithfulness：回答里的断言，有几条能在上下文里找到依据 ===
  断言「星云支持公有云部署」 → 上下文里有依据 ✅
  断言「星云支持私有化部署」 → 上下文里有依据 ✅
  断言「性能是友商两倍」 → 上下文里没有 → 判为幻觉 ❌
  算式：有依据 2 ÷ 断言总数 3 = 0.67
  （这里用"字符串包含"做教学版判定；生产上由裁判模型逐条语义判断）

=== 2) Answer Relevance：把回答反推成问题，再和原问题比相似度 ===
  （真实调用 text-embedding-v3 计算的余弦相似度）
  反推问题「星云支持哪些部署方式？」 与原问题的相似度 = 0.815
  反推问题「星云的部署方式是怎样的？」 与原问题的相似度 = 0.767
  反推问题「星云的产品价格是多少？」 与原问题的相似度 = 0.632
  算式：相似度取平均 = 0.738   ← 最后那条问的是"价格"，跟部署无关，通常最低、把均分拉下来

=== 3) Context Relevance：上下文里的句子，有几条真的跟问题相关 ===
  句子「星云支持公有云与私有化两种部署方式」 → 相关 ✅
  句子「星云成立于 2015 年」 → 跑题 ❌
  算式：相关句数 1 ÷ 句子总数 2 = 0.50
  （上面这句是另一段上下文，用来演示"整段里混进了无关句子"；相关/跑题按人工判定写死，生产上由裁判模型逐句判）

=== 3b) ContextPrecision@K：还看顺序 —— 有用的片段排得越前越高 ===
  第 1 个片段 相关=1 → Precision@1 = 1/1 = 1.00
  第 2 个片段 相关=0 → Precision@2 = 1/2 = 0.50
  第 3 个片段 相关=1 → Precision@3 = 2/3 = 0.67
  算式：ContextPrecision@3 = 1.67 ÷ 相关片段数 2 = 0.83
  对照：同样 2 条相关片段若排在第 1、2 位 → ContextPrecision@3 = 1.00（排得越靠前越值钱）


In [3]:
# 用一个极简"断言列表"思想演示 Faithfulness 判定（教学版）
# 真实实现 = LLM 把回答拆成若干断言，再逐一判断每个断言能否被上下文支持

context = ['星云支持公有云部署，星云支持私有化部署。']
answer  = '星云既支持公有云也支持私有化部署，而且性能是友商两倍。'

claims = ['星云支持公有云部署',            # 支持 → 有据
          '星云支持私有化部署',            # 支持 → 有据
          '性能是友商两倍']               # 上下文无此信息 → 幻觉

supported = sum(1 for c in claims if any(c in ctx for ctx in context))
faithfulness = supported / len(claims)
print(f'断言被支持数: {supported}/{len(claims)}')
print(f'Faithfulness = {faithfulness:.2f}  ← “性能两倍”无据导致扣分')
print('\n生产做法: 让 LLM 拆断言 → 逐条与上下文比对 → 输出 0-1 分数')

断言被支持数: 2/3
Faithfulness = 0.67  ← “性能两倍”无据导致扣分

生产做法: 让 LLM 拆断言 → 逐条与上下文比对 → 输出 0-1 分数


In [4]:
# 知识点·真调说明：Faithfulness 判定 —— 让模型把回答拆成可核验断言，再逐条对照上下文判“有据”
import json as _json
out = _llm_live(
    prompt='把下面这段“模型回答”拆成若干条可独立核验的事实断言，再对照“检索到的上下文”，'
           '逐条标记 supported=true/false（上下文没有依据就标 false）。\n'
           '上下文: 星云支持公有云部署，星云支持私有化部署。\n'
           '模型回答: 星云既支持公有云也支持私有化部署，而且性能是友商两倍。\n'
           '只输出 JSON：{"claims": [{"claim": "<断言>", "supported": true 或 false}]}',
    system='你是 RAG 忠实度(Faithfulness)评估器。判定只依据给定上下文，上下文没提到的内容一律标 false，'
           '不自行补充知识。只输出 JSON，禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"claims": [{"claim": "星云支持公有云部署", "supported": true}, '
             '{"claim": "星云支持私有化部署", "supported": true}, '
             '{"claim": "星云性能是友商两倍", "supported": false}]}',
    temperature=0.1,
)
if out is None:
    out = ('{"claims": [{"claim": "星云支持公有云部署", "supported": true}, '
           '{"claim": "星云支持私有化部署", "supported": true}, '
           '{"claim": "星云性能是友商两倍", "supported": false}]}')
    print('（以上为固定样例；下面用样例演示程序化汇总）')
try:
    data = _json.loads(out)
    claims = data['claims']
    supported = sum(1 for c in claims if c['supported'])
    print('模型把回答拆成 %d 条断言，其中 %d 条在上下文里有依据。' % (len(claims), supported))
    print('Faithfulness = %.2f' % (supported / len(claims)))
    unsup = [c['claim'] for c in claims if not c['supported']]
    if unsup:
        print('被判“无依据”的断言（幻觉）：', '；'.join(unsup))
except Exception as e:
    print('未解析成 JSON：', e, '—— 说明需在 prompt 里收紧输出格式。')
print('→ 先拆断言、再逐条核验，幻觉到底出在哪个论断一目了然——这是 RAGAS/DeepEval 计算 Faithfulness 的真实机制。')

—— 模型实时输出 ——
{"claims": [{"claim": "星云支持公有云部署", "supported": true}, {"claim": "星云支持私有化部署", "supported": true}, {"claim": "星云的性能是友商两倍", "supported": false}]}
模型把回答拆成 3 条断言，其中 2 条在上下文里有依据。
Faithfulness = 0.67
被判“无依据”的断言（幻觉）： 星云的性能是友商两倍
→ 先拆断言、再逐条核验，幻觉到底出在哪个论断一目了然——这是 RAGAS/DeepEval 计算 Faithfulness 的真实机制。


In [5]:
# LLM-as-a-Judge：让裁判模型按评分卡打分（骨架，配置 .env 后可用）
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def judge(question, context, answer):
    """让 qwen 按 3 个维度打分，输出 JSON"""
    from dashscope import Generation
    p = f"""你是 RAG 评估裁判。
问题: {question}\n上下文: {context}\n回答: {answer}
分别对 faithfulness(忠于上下文程度)、answer_relevance(切题程度)、context_relevance(上下文相关程度) 打分 0-1，只输出 JSON 如 {{"faithfulness":0.9,...}}"""
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':p}],
                        api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content

if API_KEY and '你的' not in API_KEY:
    print('裁判打分:', judge('星云支持私有化吗', ['星云支持公有云部署，星云支持私有化部署'], '支持，也支持公有云。'))
else:
    print('LLM-as-a-Judge 骨架就绪。配置 DASHSCOPE_API_KEY 后，同一套评分卡可批量评估你的 RAG。')

裁判打分: {"faithfulness":1.0,"answer_relevance":1.0,"context_relevance":1.0}


## 2. 评估框架

| 框架 | 特色 | 定位 |
|------|------|------|
| **RAGAS** | 指标体系最贴 RAG（faithfulness/context precision…） | 离线指标库 |
| **DeepEval** | pytest 风格、断言式 | 单测化评估 |
| **TruLens** | 反馈函数 + 可视化追踪 | 反馈/追踪 |
| **LangSmith** | 线上 trace + 标注 + 数据集回归 | 生产观测 |
| **RAGChecker** | 细粒度诊断（噪声敏感度等） | 深度诊断 |

> 同一套**评测集**固定后，框架只是帮你把“指标算出来”。

## 3. 开源数据集与基准

| 数据集 | 内容 | 用途 |
|--------|------|------|
| **MS MARCO** | 必应搜索真实查询+段落 | 检索/排序基准 |
| **BEIR** | 18 个异构任务合集 | 零样本泛化能力 |
| **Natural Questions** | Google 搜索问答 | 开放域问答 |
| **KILT / TriviaQA** | 知识密集任务 | 知识型 RAG |

## 小结

- 生成三指标：**Faithfulness / Answer Relevance / Context Relevance**；
- 主流做法是 **LLM-as-a-Judge**，打分卡要固定、可复现；
- 框架（RAGAS 等）与基准（MS MARCO/BEIR）解决“怎么算”“在哪比”。